## Primitive Logic --> Stateless Logic Circuits

- We've already see one level of upward abstraction in the previous section
    - PMOS/NMOS are just primitive on/off switches, but they combine to give you logical expressions
    - Logical expressions can be combined to give you novel logical behaviour (NOR + NAND + AND gives you XOR)

- Now, let's see how these primitive logic gates can get us to stateless operations! We will go through the implementations for the core primitives:
    - Adders
    - Multiplexers
    - Decoders
    - Comparators
    - Barrel Shifters & Rotators

- These primitives will give us some useful conceptual extensions, which we will also look at:
    - Subtractors
    - Multipliers
    - Encoders & Priority Encoders

- Finally, we will see how the combinations of these will jointly make up the Arithmetic Logic Unit (ALU) of a computer

### Adders

- Adders are the basic building blocks of all computer arithmetic. By chaining primitive logic gates together, we can translate Boolean logic directly into binary addition!

- There are 3 types of adders we will study
    - Half Adder
        - A half adder performs a sum of 2 bits
        - It outputs a sum bit and a carry bit
        - We consider it a half adder because it only outputs a carry bit, but doesn't accept a carry bit
    - Full Adder
        - A full adder performs the sum of 2 bits, AND a carry bit
        - It outputs a sum bit and a carry bit
    - Ripple Adder
        - This is a composite adder, which connects $N$ Full Adders in series to add $N$-bit numbers        
        - The carry output from each bit position "ripples" into the carry input of the next higher bit position

        

In [ ]:
from utils import *

def half_adder(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
            A ───┬─────────┐
                 │  ┌───┐  ├─── [cmos_XOR] ─── Sum
            B ───┼──┤XOR│──┘
                 │  └───┘
                 │  ┌───┐
                 └──┤AND│────── [cmos_AND] ─── Carry
                    └───┘
    '''
    ## - If (0, 0) --> (sum: 0, carry: 0)
    ##    - XOR(0,0) = 0, AND(0,0) = 0
    ## - If (0, 1) --> (sum: 1, carry: 0)
    ##    - XOR(0,1) = 1, AND(0,1) = 0
    ## - If (1, 0) --> (sum: 1, carry: 0)
    ##    - XOR(1,0) = 1, AND(0,1) = 0
    ## - If (1, 1) --> (sum: 0, carry: 1)
    ##    - XOR(1,1) = 0, AND(0,1) = 1

    # Returns 1 if a != b, else 0
    sum_out = cmos_XOR(a, b)

    # Returns 1 if a == b == 1, else 0
    carry_out = cmos_AND(a, b)

    return sum_out, carry_out


def full_adder(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT, c_in: TRANSISTOR_OUTPUT = GROUND) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
        A, B ──────> [Half Adder 1] ─── (Sum1, Carry1)
                           │
        Sum1, C_in ─> [Half Adder 2] ─── (Final Sum, Carry2)
                           │
        Carry1, Carry2 ─> [cmos_OR] ─── Final Carry Out

    Note that the maximum a full adder will be asked to add is 1+1+1, which means it will never exceed 3.
    This is why the output of a full adder can simply be a sum, carry
    The sum can only be 0 or 1, since it is the sum of 2 binary values
    The carry can represent a maximum of the 2^1 (since it is a carry, it represents the binary value for the 
    next significant position)

    - (0, 0, 0) -> (0, 0)
        - HA(a=0, b=0) = s1=0, c1=0
        - HA(s1=0, c=0) = s2=0, c2=0
        - OR(c1=0, c=0) = c3=0
        - Output(s2=0, c3=0)
    - (0, 0, 1) | (0, 1, 0) | (1, 0, 0) -> (1, 0)
        - HA(a=1, b=0) = s1=1, c1=0
        - HA(s1=1, c=0) = s2=1, c2=0
        - OR(c1=0, c2=0) = c3=0
        - Output(s2=1, c3=0)
    - (0, 1, 1) | (1, 1, 0) | (1, 0, 1) -> (0, 1)
        - HA(a=1, b=1) = s1=0, c1=1
        - HA(s1=0, c=0) = s2=0, c2=0
        - OR(c1=1, c2=0) = c3=1
        - Output(s2=0, c3=1)
    - (1, 1, 1) -> (1, 1)
        - HA(a=1, b=1) = s1=0, c1=1
        - HA(s1=0, c=1) = s2=1, c2=0
        - OR(c1=1, c2=0) = c3=1
        - Output(s2=1, c3=1)
    '''
    
    
    sum1, carry1 = half_adder(a, b)

    final_sum, carry2 = half_adder(sum1, c_in)
    
    final_carry = cmos_OR(carry1, carry2)
    return final_sum, final_carry


def ripple_carry_adder(a_bits: list[TRANSISTOR_OUTPUT], b_bits: list[TRANSISTOR_OUTPUT]) -> tuple[list[TRANSISTOR_OUTPUT], TRANSISTOR_OUTPUT]:
    '''
    Ripple-Carry Adder processing LSB to MSB.
    Takes two equal-length lists of bit signals (ordered LSB -> MSB).
    Returns (sum_bits, final_carry_out)

        LSB (Bit 0)                 Bit 1                    MSB (Bit 2)
        a_bits[0] = 1            a_bits[1] = 1              a_bits[2] = 1
        b_bits[0] = 1            b_bits[1] = 1              b_bits[2] = 1
                │                        │                          │
                │   ┌─────────┐          │   ┌─────────┐            │   ┌─────────┐
                ├───┤         │          ├───┤         │            ├───┤         │
                │   │ Full    │          │   │ Full    │            │   │ Full    │
                └───┤ Adder 0 │          └───┤ Adder 1 │            └───┤ Adder 2 │
                    │         │              │         │                │         │
    GROUND ─────────┤ c_in    │  ┌───────────┤ c_in    │    ┌───────────┤ c_in    │
    (c_in = 0)      │         │  │ (c_out=1) │         │    │ (c_out=1) │         │
                    │   c_out ├──┘           │   c_out ├────┘           │   c_out ├─── Final Carry
                    │   sum   ├──┐           │   sum   ├──┐             │   sum   ├──┐ (carry = 1)
                    └─────────┘  │           └─────────┘  │             └─────────┘  │
                                ▼                        ▼                          ▼
                            sum_bits[0]              sum_bits[1]                sum_bits[2]
                            (Sum = 0)                (Sum = 1)                  (Sum = 1)

    
    ============
       EXAMPLE
    ============
    - a_bits = 5 = [1,0,1]
    - b_bits = 3 = [0,1,1]
    - carry = 0
    - sum_bits = []
    
    - Going from LSB to MSB:
        - full_adder(a=1, b=1, c=0) 
            - s1=0, c1=1
            - sum_bits = [,0]
        - full_adder(a=0, b=1, c=c1=1)
            - s2=0, c2=1
            - sum_bits = [,0,0]
        - full_adder(a=1, b=0, c=c2=1)
            - s3=0, c3=1
            - sum_bits = [0,0,0]
    
    - Final value:
        - Concatenate sum_bits and prepend the final carry
        - output = 1000 = 8 = 5 + 3

    '''
    carry: TRANSISTOR_OUTPUT = GROUND
    sum_bits: list[TRANSISTOR_OUTPUT] = []
    
    for bit_a, bit_b in zip(a_bits, b_bits):
        s, carry = full_adder(bit_a, bit_b, carry)
        sum_bits.append(s)
        
    return sum_bits, carry